# Libs

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

%env TF_ENABLE_ONEDNN_OPTS=0

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kurtosis, skew, entropy, linregress
from collections import defaultdict

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN, SpectralClustering, AgglomerativeClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE, MDS, Isomap
from sklearn.decomposition import PCA, KernelPCA
from sklearn.impute import KNNImputer
from sklearn.metrics import euclidean_distances

from umap import UMAP

from tsfresh.feature_extraction import extract_features, EfficientFCParameters
from tsfresh.utilities.dataframe_functions import impute

import futurai_ppd as ppd
import futurai_utils as utils
from futurai_ml_dev import FuturaiML

DIR_DATA = os.getcwd() + "/data/"

## Functions

In [ ]:
def localizar_periodos_com_anomalia(futurai, result):
    df_pred = pd.DataFrame({"timestamp": result["timestamp"], "phi": result["phi"]})
    df_pred["timestamp"] = pd.to_datetime(df_pred["timestamp"], format="%Y-%m-%d %H:%M:%S")
    df_pred = df_pred.set_index('timestamp')
    df_pred = df_pred.resample("1T").asfreq()
    df_pred = df_pred.fillna(0)
    df_pred.reset_index(inplace=True)
    data_max = df_pred["timestamp"].max()
    data_min = df_pred["timestamp"].min()
    data_aux = data_min

    ##### onde se localiza a anomalia e preenche lista com datas de começo(onde phi>lim) e fim (onde phi<lim) #####
    list_datas: list[list[datetime, datetime]] = list()
    while data_aux <= data_max:
        mask = (df_pred["timestamp"] >= data_aux) & (
            df_pred["phi"] > futurai.phi_lim
        )
        df_aux2 = df_pred.loc[mask]
        if not df_aux2.empty:
            data_comeco = df_aux2["timestamp"].min()
        else:
            break

        mask = (df_pred["timestamp"] >= data_comeco) & (
            df_pred["phi"] <= futurai.phi_lim
        )
        df_aux3 = df_pred.loc[mask]
        
        if not df_aux3.empty:
            data_depois = df_aux3["timestamp"].min()
            data_aux = data_depois
        else:
            data_depois = df_aux2["timestamp"].max()
            list_datas.append([data_comeco, data_depois])
            break
            
        list_datas.append([data_comeco, data_depois])
    return list_datas

#########################################################################################################################################################
def unificar_anomalias(list_datas, anomaly_interval):
    list_datas_copy = list_datas.copy()
    dir = list_datas_copy[0][0]
    dfa = list_datas_copy[0][1]
    list_datas_copy.pop(0)

    list_datas_final = []

    anomalias_unificadas = [[(dir, dfa)]]
    idx_anomalias_unificadas = 0
    for per in list_datas_copy:
        di = per[0]
        df = per[1]
        interval = (di - dfa).total_seconds() / 60
        if interval < 0:
            pass
        if interval < anomaly_interval:
            dfa = df

            anomalias_unificadas[idx_anomalias_unificadas].append((di, df))
        else:
            list_datas_final.append([dir, dfa])
            dir = di
            dfa = df

            anomalias_unificadas.append(list())
            idx_anomalias_unificadas += 1
            anomalias_unificadas[idx_anomalias_unificadas].append((di, df))
    list_datas_final.append([dir, dfa])
    list_datas_final.reverse()
    
    dict_anomalias_unificadas = dict()
    date_fmt = '%Y-%m-%d %H:%M:%S'
    for anomalia_unificada in anomalias_unificadas:
        num_anomalias = len(anomalia_unificada)
        if num_anomalias <= 1:
            continue
        di = anomalia_unificada[0][0]
        df = anomalia_unificada[-1][1]
        di_str = di.strftime(date_fmt)
        df_str = df.strftime(date_fmt)
        key = f"{di_str} - {df_str}"
        dict_anomalias_unificadas.update({key: anomalia_unificada})

    return list_datas_final, dict_anomalias_unificadas

#########################################################################################################################################################
def periods_above_threshold(df_data, variable_datetime, predictions, model):
    """
    Identify periods in the DataFrame where values exceed the threshold.

    Args:
        df_data (pd.DataFrame): The input DataFrame with variables values.
        predictions (dict): The result dictionary containing predictions results and timestamps.
        model (object): The model object containing the threshold value.

    Returns:
        pd.DataFrame: A DataFrame containing only the periods above the threshold.
    """
    phi_values = np.array(predictions['phi'])
    indexes = np.where(phi_values > model.phi_lim)[0]
    indexes_expand = []
    window=1
    for idx in indexes:
        for i in range(idx - window, idx + window + 1):
            if 0 <= i < len(phi_values):
                indexes_expand.append(i)
    indexes_expand = sorted(set(indexes_expand))
    timestamps_above_threshold = np.array(predictions['timestamp'])[indexes_expand]

    df_data[variable_datetime] = pd.to_datetime(df_data[variable_datetime])
    timestamps_above_threshold = pd.to_datetime(timestamps_above_threshold)

    df_filtered = df_data[df_data[variable_datetime].isin(timestamps_above_threshold)]

    return df_filtered

#########################################################################################################################################################
def extract_tsfresh_features_robust(df_anomaly, anomaly_id=0):
    """
    Extrai features robustas para diagnóstico industrial usando tsfresh.
    
    Args:
        df_anomaly (pd.DataFrame): DataFrame com índice temporal e colunas de sensores.
        anomaly_id (int/str): Identificador da anomalia (útil se for processar em lote depois).
        
    Returns:
        dict: Dicionário com as features extraídas e tratadas (sem NaNs).
    """
    
    # 1. Validação Básica
    if df_anomaly.empty:
        return {}

    # 2. Preparação dos Dados (Wide -> Long Format)
    # O tsfresh exige formato: [id, time, kind (sensor), value]
    df_process = df_anomaly.copy()
    
    # Se o índice não tiver nome, damos um nome padrão para poder resetar
    if df_process.index.name is None:
        df_process.index.name = 'timestamp'
        
    df_long = (
        df_process
        .reset_index()
        .melt(id_vars=df_process.index.name, var_name='kind', value_name='value')
    )
    
    # Adiciona ID único para este trecho de dados
    df_long['id'] = anomaly_id
    
    # Garante que o tempo está no formato correto para ordenação
    time_col = df_process.index.name
    
    # 3. Definição do Dicionário de Features (Industrial Settings)
    # Ajustei a sintaxe para garantir compatibilidade total
    industrial_fc_parameters = {
        # --- Estatísticas Básicas (Amplitude) ---
        "mean": None,
        "median": None,
        "standard_deviation": None, # Variância removida (redundante)
        "minimum": None,
        "maximum": None,
        "root_mean_square": None, # Ouro para vibração
        
        # --- Dinâmica e Tendência (Degradação) ---
        "mean_abs_change": None, # Volatilidade
        # agg_linear_trend é mais robusto que linear_trend simples para ruído
        "agg_linear_trend": [{"attr": "slope", "chunk_len": 50, "f_agg": "mean"}],
        
        # --- Forma da Distribuição (Detecção de Outliers/Impactos) ---
        "skewness": None, # Assimetria (ótimo para falhas unilaterais)
        "kurtosis": None, # Achatamento (ótimo para impactos/batidas)
        "quantile": [{"q": 0.05}, {"q": 0.95}], # Foco nas caudas extremas
        
        # --- Comportamento Temporal & Complexidade (O que faltava) ---
        "binned_entropy": [{"max_bins": 10}], # Mede o "caos" ou saúde do sistema
        "longest_strike_above_mean": None,
        "number_peaks": [{"n": 3}, {"n": 5}], # Contagem de picos locais
        
        # --- Frequência Avançada (Densidade Espectral) ---
        # Substitui coeficientes FFT crus por PSD (Power Spectral Density)
        # Captura energia em diferentes bandas de frequência
        "spkt_welch_density": [{"coeff": 2}, {"coeff": 5}, {"coeff": 8}], 
        
        # Zero crossing rate (importante para oscilação mecânica)
        "count_above_mean": None, 
    }

    # 4. Extração
    try:
        # n_jobs=0 evita overhead de multiprocessamento para dataframes pequenos
        X = extract_features(
            df_long,
            column_id="id",
            column_sort=time_col,
            column_kind="kind",
            column_value="value",
            default_fc_parameters=industrial_fc_parameters,
            disable_progressbar=True,
            n_jobs=0 
        )
        
        # 5. Imputação (CRUCIAL)
        # Substitui NaNs (gerados por divisões por zero ou séries constantes) pela mediana
        X = impute(X)
        
        # Retorna como dicionário plano
        return X.iloc[0].to_dict()

    except Exception as e:
        print(f"Erro na extração de features para ID {anomaly_id}: {e}")
        return {}

#########################################################################################################################################################
def extract_features_manually(df_anomaly):
    features = {}
    for col in df_anomaly.columns:
        # Standard Features
        features[f"{col}_mean"] = df_anomaly[col].mean()
        features[f"{col}_std"] = df_anomaly[col].std()
        features[f"{col}_min"] = df_anomaly[col].min()
        features[f"{col}_max"] = df_anomaly[col].max()
        features[f"{col}_range"] = df_anomaly[col].max() - df_anomaly[col].min()
        features[f"{col}_cv"] = df_anomaly[col].std() / df_anomaly[col].mean() if df_anomaly[col].mean() != 0 else np.nan
        
        # Estatísticas de forma
        features[f"{col}_skew"] = skew(df_anomaly[col], bias=False)
        features[f"{col}_kurtosis"] = kurtosis(df_anomaly[col], bias=False)
        
        # Energia (soma dos quadrados)
        features[f'{col}_energy'] = np.sum(np.square(df_anomaly[col]))
        # RMS (Root Mean Square)
        features[f'{col}_rms'] = np.sqrt(np.mean(np.square(df_anomaly[col])))
        # Entropia de Shannon
        hist, bin_edges = np.histogram(df_anomaly[col], bins='auto', density=True)
        hist = hist[hist > 0]  # remove zeros
        features[f'{col}_entropy'] = entropy(hist)
    return features


#########################################################################################################################################################
def visualize_tsne(
    X_scaled,
    labels=None,
    perplexity=30.0,
    learning_rate='auto',
    max_iter=1000,
    random_state=42,
    method='barnes_hut',
    angle=0.5,
    metric='euclidean',
    verbose=1
):
    """
    Visualiza dados de alta dimensão usando t-SNE (scikit-learn >= 1.4 compatível).
    
    Parâmetros:
    -----------
    X_scaled : array-like, shape (n_samples, n_features)
        Dados normalizados.
    labels : array-like, opcional
        Rótulos (clusters ou classes) para colorir os pontos.
    perplexity : float
        Controla o equilíbrio entre vizinhanças locais e globais (típico: 5–50).
    learning_rate : float ou 'auto'
        Taxa de aprendizado (200–1000 normalmente).
    max_iter : int
        Número máximo de iterações de otimização.
    random_state : int
        Semente para reprodutibilidade.
    method : str
        'barnes_hut' (rápido, padrão) ou 'exact' (mais preciso).
    angle : float
        Parâmetro de trade-off velocidade/precisão no método 'barnes_hut' (0.2–0.8).
    metric : str
        Métrica de distância (ex: 'euclidean', 'cosine', 'manhattan').
    verbose : int
        0 = silencioso | 1 = imprime progresso.

    Retorna:
    --------
    np.ndarray
        Coordenadas 2D projetadas pelo t-SNE.
    """
    if verbose:
        print(f"Executando t-SNE (perplexity={perplexity}, learning_rate={learning_rate}, max_iter={max_iter})...")

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        early_exaggeration=12.0,
        learning_rate=learning_rate,
        max_iter=max_iter,
        n_iter_without_progress=500,
        min_grad_norm=1e-7,
        metric=metric,
        init='pca',
        random_state=random_state,
        method=method,
        angle=angle,
        verbose=verbose,
        n_jobs=None
    )

    X_embedded = tsne.fit_transform(X_scaled)

    if verbose:
        print("t-SNE concluído!")

    # --- Plot 2D ---
    plt.figure(figsize=(8, 6))
    if labels is not None:
        n_labels = len(np.unique(labels))
        palette = sns.color_palette("husl", n_labels if n_labels > 1 else 2)
        sns.scatterplot(
            x=X_embedded[:, 0],
            y=X_embedded[:, 1],
            hue=labels,
            palette=palette,
            s=40,
            alpha=0.85,
            edgecolor='none'
        )
        plt.legend(title="Cluster / Classe", bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        plt.scatter(X_embedded[:, 0], X_embedded[:, 1], s=40, alpha=0.7, c='red')

    plt.title("Visualização com t-SNE")
    plt.xlabel("Componente 1 (t-SNE)")
    plt.ylabel("Componente 2 (t-SNE)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return X_embedded

#########################################################################################################################################################

def visualize_umap(
    X_scaled,
    labels=None,
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric='euclidean',
    random_state=42,
    verbose=True
):
    """
    Visualiza dados de alta dimensão usando UMAP (Uniform Manifold Approximation and Projection).
    
    Geralmente é mais rápido que o t-SNE e preserva melhor a estrutura global dos dados.
    
    Parâmetros:
    -----------
    X_scaled : array-like, shape (n_samples, n_features)
        Dados normalizados (Features extraídas ou PCA).
    labels : array-like, opcional
        Rótulos (clusters ou classes) para colorir os pontos.
    n_neighbors : int
        Equivalente ao 'perplexity' do t-SNE. Controla o tamanho da vizinhança local.
        - Valores baixos (2-10): Foca em estrutura local (detalhes finos).
        - Valores altos (30-100): Foca em estrutura global (visão geral).
    min_dist : float
        Distância mínima entre pontos no espaço projetado (0.0 a 0.99).
        - Baixo (ex: 0.1): Clusters mais compactos e aglomerados.
        - Alto (ex: 0.5): Pontos mais espalhados, preserva melhor a topologia.
    n_components : int
        Dimensão final (padrão 2 para visualização).
    metric : str
        Métrica de distância (ex: 'euclidean', 'manhattan', 'cosine').
    random_state : int
        Semente para reprodutibilidade.
    verbose : bool
        Se True, imprime o progresso.

    Retorna:
    --------
    np.ndarray
        Coordenadas 2D projetadas pelo UMAP.
    """
    
    if verbose:
        print(f"Executando UMAP (n_neighbors={n_neighbors}, min_dist={min_dist}, metric={metric})...")

    # Instancia o redutor UMAP
    reducer = UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric,
        random_state=random_state,
        verbose=verbose
    )

    # Ajusta e transforma os dados
    X_embedded = reducer.fit_transform(X_scaled)

    if verbose:
        print("UMAP concluído!")

    # --- Plot 2D (Mantendo o estilo do seu código original) ---
    plt.figure(figsize=(8, 6))
    
    if labels is not None:
        n_labels = len(np.unique(labels))
        # Paleta segura para evitar erro se houver apenas 1 cluster
        palette = sns.color_palette("husl", n_labels if n_labels > 1 else 2)
        
        sns.scatterplot(
            x=X_embedded[:, 0],
            y=X_embedded[:, 1],
            hue=labels,
            palette=palette,
            s=40,
            alpha=0.85,
            edgecolor='none'
        )
        # Posiciona a legenda fora do gráfico para não cobrir dados
        plt.legend(title="Cluster / Classe", bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        plt.scatter(X_embedded[:, 0], X_embedded[:, 1], s=40, alpha=0.7, c='blue')

    plt.title(f"Visualização com UMAP (neighbors={n_neighbors}, min_dist={min_dist})")
    plt.xlabel("Componente 1 (UMAP)")
    plt.ylabel("Componente 2 (UMAP)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return X_embedded

#########################################################################################################################################################

def plot_variable_by_cluster(df_data, labels, variable_name, timestamp_col='timestamp'):
    """
    Plota o comportamento de uma variável específica para cada cluster.
    
    Args:
        df_data: DataFrame 'df_dataset_above_threshold' (com coluna 'anomaly_id').
        labels: Lista com os labels dos clusters (na ordem dos anomaly_ids).
        variable_name: Nome da coluna/variável que você quer analisar.
        timestamp_col: Nome da coluna de tempo (para eixo X).
    """
    
    cluster_map = {i: label for i, label in enumerate(labels)}
    
    df_plot = df_data.copy()
    df_plot['cluster'] = df_plot['anomaly_id'].map(cluster_map)
    
    unique_clusters = sorted(list(set(labels)))
    n_clusters = len(unique_clusters)
    
    fig, axes = plt.subplots(n_clusters, 1, figsize=(12, 4 * n_clusters), sharex=False)
    if n_clusters == 1: axes = [axes]
    
    for ax, cluster_id in zip(axes, unique_clusters):
        
        df_cluster = df_plot[df_plot['cluster'] == cluster_id]
        anom_ids = df_cluster['anomaly_id'].unique()
        
        for a_id in anom_ids:
            subset = df_cluster[df_cluster['anomaly_id'] == a_id]
            
            x_axis = np.arange(len(subset))
            
            ax.plot(x_axis, subset[variable_name].values, alpha=0.3, color='blue', linewidth=1)
            
        # Adiciona a "Média" do cluster (linha mais grossa) para referência visual
        # Nota: Isso exige reamostragem se os tamanhos forem diferentes, aqui é apenas visual
        
        ax.set_title(f"Cluster {cluster_id} - Variável: {variable_name} ({len(anom_ids)} eventos)")
        ax.set_xlabel("Tempo Relativo (amostras)")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Load Data

In [ ]:
process_name = "Refinador - Rosca Extração"
timestamp="TIMESTAMP"

list_colomuns_drop = ["espessura_receita_prensa"]

df_dataset = ppd.load_dataset_principal(DIR_DATA+process_name+".csv", list_colomuns_drop, timestamp, dropna=True, use_chunks=True, chunksize=10000)
df_dataset

# Set ON/OFF var

In [ ]:
pre_process = []
pp_var_ref_desligado = "R_2314S_Motor"
pp_valor_ref_desligado = 400
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 60
pp_pos_corte_transitorio = 210
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

# TAGs and descriptions

In [ ]:
df_sistema, df_sistema_drop =  ppd.set_tags_config(df_dataset,DIR_DATA+process_name+"_subsistema.csv")
df_sistema

# Training periods

## Period 1

In [ ]:
# train 1
start_date_train = pd.to_datetime("2025-07-10 00:00:00")
end_date_train = pd.to_datetime("2025-07-15 00:00:00")

mask = (df_dataset[timestamp] >= start_date_train) & (df_dataset[timestamp] <= end_date_train)
df_train = df_dataset.loc[mask]

# pre - processamento
print(df_train.shape)
list_periods_train = []
for pro in pre_process:
    df_train,_,list_aux = ppd.drop_transitorio_desligado(df_train,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]
df_train.reset_index(inplace=True, drop=True)
print(df_train.shape)

eixoX_train = df_train[timestamp]
df_train = df_train.drop(timestamp,axis=1)

## Period 2

In [ ]:
#train 2
start_date_train2 = pd.to_datetime("2025-10-21 20:00:00")
end_date_train2 = pd.to_datetime("2025-10-23 09:00:00")

mask = (df_dataset[timestamp] > start_date_train2) & (df_dataset[timestamp] <= end_date_train2)
df_train2 = df_dataset.loc[mask]

# pre - processamento
print(df_train2.shape)
list_periods_train = []
for pro in pre_process:
    df_train2,_,list_aux = ppd.drop_transitorio_desligado(df_train2,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]
df_train2.reset_index(inplace=True, drop=True)
print(df_train2.shape)

eixoX_train2 = df_train2[timestamp]
df_train2 = df_train2.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train2], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train2], ignore_index=True)

# Fit Model

In [ ]:
# Instacinamento da Classe
gain = 2.5
nc = 0
futurai = FuturaiML(nc,gain)

# Gerando o modelo
futurai.fit(df_train)

# Print dos limiares do modelo
print("Modelo")
print("T²: {:.2f}".format(futurai.t2_lim))
print("Q: {:.2f}".format(futurai.q_lim))
print("Phi: {:.2f}".format(futurai.phi_lim))
print("Componentes: {:}".format(futurai.nc))     

# Predict

In [ ]:
start_date = pd.to_datetime("2025-08-01 00:00:00")
end_date = pd.to_datetime("2025-12-30 19:21:00")

mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

# pre - processamento
print(df_test.shape)
list_periods_test = []
for pro in pre_process:
    df_test,_,list_aux = ppd.drop_transitorio_desligado(df_test,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_test = [*list_periods_test,*list_aux]
print(df_test.shape)

eixoX_test = df_test[timestamp]
df_test_aux = df_test.copy()
df_test_aux.set_index(timestamp, inplace=True)
df_test = df_test.drop(timestamp,axis=1)
df_test.reset_index(inplace=True, drop=True)

result = futurai.predict(df_test,eixoX_test)

## Plot predictions

In [ ]:
list_periods_test = ppd.merge_periods(list_periods_test)
fig_all_period, _ = utils.dev_graph_predict(result["phi"], result["timestamp"], futurai.phi_lim, " ", start_date, end_date, list_periods=False, plot_anomalies=False)
fig_all_period.show()

# Locate Anomalies

In [ ]:
list_datas = localizar_periodos_com_anomalia(futurai, result)
anomaly_interval = 1440
anomaly_periods, _ = unificar_anomalias(list_datas, anomaly_interval)
print(f"Qtd. anomalias: {len(anomaly_periods)}")
anomaly_periods.reverse()

# Calculate anomalies scores

In [ ]:
df_scores_list = []
for anomaly in anomaly_periods:
    mask = (df_dataset[timestamp] >= anomaly[0]) & (df_dataset[timestamp] <= anomaly[1])
    df_anom = df_dataset.loc[mask]

    for pro in pre_process:
        df_anom,_,_ = ppd.drop_transitorio_desligado(df_anom,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])

    df_anom_with_timestamp = df_anom.copy()
    eixoX_anom = df_anom[timestamp]
    df_anom = df_anom.drop(timestamp,axis=1)
    df_anom.reset_index(inplace=True, drop=True)
    result = futurai.predict(df_anom,eixoX_anom)

    df_filtered = periods_above_threshold(df_anom_with_timestamp, timestamp, result, futurai)
    eixoX_anom_filtered = df_filtered[timestamp]
    df_filtered.drop(timestamp, axis=1, inplace=True)
    df_filtered.reset_index(inplace=True, drop=True)
    result = futurai.predict(df_filtered, eixoX_anom_filtered)
    _, _, dict_full, _ = futurai.contribuition(df_filtered,result["matrix"], df_sistema, result["timestamp"], eixoX_anom_filtered)
    df_full = pd.DataFrame().from_dict(dict_full)
    df_row = df_full.set_index('VARIAVEL')[['score']].T
    df_row.index.name = None
    df_scores_list.append(df_row)

if df_scores_list:
    df_scores = pd.concat(df_scores_list, ignore_index=True)
else:
    df_scores = pd.DataFrame()

df_scores

In [ ]:
scaler = StandardScaler()
X_scores_normalized = scaler.fit_transform(df_scores)
X_scores_normalized.shape

# Clustering

## Feature based

### Visualize Data

In [ ]:
X_visualize = X_scores_normalized  ## X_all_features_scaled | X_wavelet_features

#### t-SNE on high dimensional dataset

In [ ]:
X_tsne = visualize_tsne(X_visualize, labels=None, perplexity=5, learning_rate='auto', max_iter=10000, random_state=42, method='exact', angle=0.5, metric='euclidean', verbose=0)

#### MDS on high dimensional dataset

In [ ]:
mds = MDS(n_components=2, n_init=5, random_state=42)
X_mds = mds.fit_transform(X_visualize)
print(X_mds.shape)

plt.figure(figsize=(8, 6))
plt.scatter(X_mds[:, 0], X_mds[:, 1], s=40, alpha=0.7, c='red')
plt.title("Visualização com MDS")
plt.xlabel("Componente 1 (MDS)")
plt.ylabel("Componente 2 (MDS)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

#### UMAP on high dimensional dataset

In [ ]:
X_umap = visualize_umap(X_visualize, labels=None, n_neighbors=5, min_dist=0.1, n_components=2, metric='euclidean', random_state=42, verbose=False)

### PCA to dimensionality reduction

In [ ]:
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scores_normalized)
nc = X_pca.shape[1]
print(f"Número de componentes principais: {nc}")

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=40, alpha=0.7, c='red')
plt.title("Visualização com PCA")
plt.xlabel("Componente 1 (PCA)")
plt.ylabel("Componente 2 (PCA)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Define parameters to run algorithms

In [ ]:
X_cluster = X_scores_normalized ## X_pca || X_scores_normalized

### K-Means

In [ ]:
def find_optimal_k_kmeans(X_scaled, k_min=1, k_max=10, plot=True):
    """
    Busca automática do número ótimo de clusters usando várias métricas de validação.
    
    Calcula:
    - Silhouette
    - Calinski-Harabasz
    - Inertia (Elbow)

    Exibe todos os gráficos em um único subplot.

    Returns:
    --------
    dict:
        scores (dict com listas de scores),
        best_models (dict com modelos KMeans para cada métrica)
    """

    ks = list(range(k_min, k_max + 1))

    # Armazena métricas
    results = {
        "silhouette": [],
        "calinski": [],
        "inertia": []
    }

    # Para salvar o modelo final por métrica
    best_models = {}

    for k in ks:
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X_scaled)

        inertia = kmeans.inertia_
        results["inertia"].append(inertia)

        # Silhouette requer K >= 2
        if k > 1:
            results["silhouette"].append(silhouette_score(X_scaled, labels))
            results["calinski"].append(calinski_harabasz_score(X_scaled, labels))
        else:
            results["silhouette"].append(np.nan)
            results["calinski"].append(np.nan)

    # -----------------------------
    # Determinação de cada K ótimo
    # -----------------------------
    best_k = {}

    best_k["silhouette"] = ks[np.nanargmax(results["silhouette"])]
    best_k["calinski"]   = ks[np.argmax(results["calinski"])]

    # Para o método do cotovelo:
    diffs = np.diff(results["inertia"])
    second_diffs = np.diff(diffs)
    elbow_k = np.argmin(second_diffs) + k_min + 1
    best_k["elbow"] = elbow_k

    # Modelos finais
    for metric_name, k_value in best_k.items():
        best_models[metric_name] = KMeans(n_clusters=k_value, random_state=42).fit(X_scaled)

    # -----------------------------
    # Plot em subplots
    # -----------------------------
    if plot:

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes.ravel()

        # --- Silhouette ---
        ax1.plot(ks, results["silhouette"], marker='o')
        ax1.set_title("Silhouette Score")
        ax1.set_xlabel("K")
        ax1.set_ylabel("Silhouette")
        ax1.grid()

        # --- Calinski-Harabasz ---
        ax2.plot(ks, results["calinski"], marker='o')
        ax2.set_title("Calinski-Harabasz")
        ax2.set_xlabel("K")
        ax2.set_ylabel("Score")
        ax2.grid()

        # --- Elbow (Inertia) ---
        ax3.plot(ks, results["inertia"], marker='o')
        ax3.set_title("Método do Cotovelo (Inertia)")
        ax3.set_xlabel("K")
        ax3.set_ylabel("Inertia")
        ax3.grid()

        plt.tight_layout()
        plt.show()

    # Retorno organizado
    return {
        "best_k": best_k,
        "scores": results,
        "best_models": best_models
    }

In [ ]:
result_kmeans = find_optimal_k_kmeans(X_cluster, k_min=1, k_max=10) #silhouette | elbow | calinski

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42)
labels_kmeans = kmeans.fit_predict(X_cluster)

In [ ]:
X_umap = visualize_umap(X_cluster, labels=labels_kmeans, n_neighbors=5, min_dist=0.1, n_components=2, metric='euclidean', random_state=42, verbose=False)

In [ ]:
X_tsne = visualize_tsne(X_cluster, labels=labels_kmeans, perplexity=5, learning_rate='auto', max_iter=10000, random_state=42, method='exact', angle=0.5, metric='euclidean', verbose=0)

#### Plot variables by clusters

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_kmeans, 
    variable_name='torque_ds', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_kmeans, 
    variable_name='L1_2316', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_kmeans, 
    variable_name='PIC2316_CV', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

In [ ]:
cluster_colors = {
    0: 'rgba(31, 119, 180, 1)',   # Azul Forte
    1: 'rgba(255, 127, 14, 1)',   # Laranja Vivo
    2: 'rgba(44, 160, 44, 1)',    # Verde Floresta
    3: 'rgba(214, 39, 40, 1)',    # Vermelho Tijolo
    4: 'rgba(148, 103, 189, 1)',  # Roxo
    5: 'rgba(140, 86, 75, 1)',    # Marrom
    6: 'rgba(227, 119, 194, 1)',  # Rosa
    7: 'rgba(127, 127, 127, 1)',  # Cinza
    8: 'rgba(188, 189, 34, 1)',   # Verde Oliva
    9: 'rgba(23, 190, 207, 1)'    # Azul Teal/Ciano
}

for (start, end), label in zip(anomaly_periods, labels_kmeans):
    color = cluster_colors.get(label, 'rgba(128, 128, 128, 0.3)')
    
    fig_all_period.add_vrect(
        x0=start, 
        x1=end,
        fillcolor=color,
        opacity=1,           # Opacidade já controlada na cor (rgba) ou use float aqui se usar hex
        layer="below",       # Coloca a faixa atrás das linhas do gráfico
        line_width=0,        # Remove borda da faixa
    )

fig_all_period.show()

### Hierachical Clustering

In [ ]:
import scipy.cluster
import scipy.cluster.hierarchy as hierarchy

Z = hierarchy.linkage(X_cluster, method='ward')
hierarchy.dendrogram(Z, color_threshold=0);

In [ ]:
def find_optimal_k_hierarchical(X_scaled, k_min=2, k_max=10, plot=True):
    """
    Busca automática do K ótimo para Hierarchical Clustering (Agglomerative).
    
    Calcula:
    - Silhouette
    - Calinski-Harabasz
    - Davies-Bouldin
    - Pseudo-Inertia (WSS - Soma dos Quadrados Intra-Cluster)

    Args:
        k_min (int): Mínimo de clusters (>= 2)
        k_max (int): Máximo de clusters
    """

    ks = list(range(k_min, k_max + 1))

    # Armazena métricas
    results = {
        "silhouette": [],
        "calinski": [],
        "davies": [],
        "wss": [] # "Within-Cluster Sum of Squares" (Pseudo-Inertia)
    }

    # Para salvar o modelo final por métrica
    best_models = {}

    for k in ks:
        # linkage='ward' minimiza a variância dos clusters sendo fundidos
        hc = AgglomerativeClustering(n_clusters=k, linkage='ward')
        labels = hc.fit_predict(X_scaled)

        # Métricas de validação
        results["silhouette"].append(silhouette_score(X_scaled, labels))
        results["calinski"].append(calinski_harabasz_score(X_scaled, labels))
        results["davies"].append(davies_bouldin_score(X_scaled, labels))

        # --- Cálculo Manual da WSS (Para simular o Cotovelo) ---
        # Necessário pois AgglomerativeClustering não expõe .inertia_
        wss_val = 0
        for cluster_id in range(k):
            cluster_points = X_scaled[labels == cluster_id]
            if len(cluster_points) > 0:
                centroid = cluster_points.mean(axis=0)
                wss_val += np.sum((cluster_points - centroid) ** 2)
        results["wss"].append(wss_val)

    # -----------------------------
    # Determinação de cada K ótimo
    # -----------------------------
    best_k = {}

    best_k["silhouette"] = ks[np.argmax(results["silhouette"])]
    best_k["calinski"]   = ks[np.argmax(results["calinski"])]
    best_k["davies"]     = ks[np.argmin(results["davies"])]
    
    # Método do Cotovelo (simplificado na segunda derivada da WSS)
    diffs = np.diff(results["wss"])
    second_diffs = np.diff(diffs)
    elbow_idx = np.argmin(second_diffs) 
    best_k["elbow"] = ks[elbow_idx + 1] if (elbow_idx + 1) < len(ks) else ks[elbow_idx]

    # Modelos finais
    for metric_name, k_value in best_k.items():
        best_models[metric_name] = AgglomerativeClustering(n_clusters=k_value, linkage='ward').fit(X_scaled)

    # -----------------------------
    # Plot em subplots
    # -----------------------------
    if plot:
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes.ravel()

        # --- Silhouette ---
        ax1.plot(ks, results["silhouette"], marker='o', color='blue')
        ax1.set_title("Silhouette Score")
        ax1.set_xlabel("K")
        ax1.set_ylabel("Silhouette")
        ax1.grid()

        # --- Calinski-Harabasz ---
        ax2.plot(ks, results["calinski"], marker='o', color='green')
        ax2.set_title("Calinski-Harabasz")
        ax2.set_xlabel("K")
        ax2.set_ylabel("Score")
        ax2.grid()

        # --- Davies-Bouldin ---
        ax3.plot(ks, results["davies"], marker='o', color='red')
        ax3.set_title("Davies-Bouldin (Menor é melhor)")
        ax3.set_xlabel("K")
        ax3.set_ylabel("Score")
        ax3.grid()

        # --- WSS (Elbow) ---
        ax4.plot(ks, results["wss"], marker='o', color='purple')
        ax4.set_title("Pseudo-Inertia / WSS (Método do Cotovelo)")
        ax4.set_xlabel("K")
        ax4.set_ylabel("Soma Quadrados Distância")
        ax4.grid()

        plt.tight_layout()
        plt.show()

    return {
        "best_k": best_k,
        "scores": results,
        "best_models": best_models
    }

In [ ]:
resultado_hc = find_optimal_k_hierarchical(X_cluster, k_min=2, k_max=10)

In [ ]:
hc = AgglomerativeClustering(n_clusters=2, linkage='ward')
labels_hierarchical = hc.fit_predict(X_cluster)

In [ ]:
X_umap = visualize_umap(X_cluster, labels=labels_hierarchical, n_neighbors=5, min_dist=0.1, n_components=2, metric='euclidean', random_state=42, verbose=False)

In [ ]:
X_tsne = visualize_tsne(X_cluster, labels=labels_hierarchical, perplexity=5, learning_rate='auto', max_iter=10000, random_state=42, method='exact', angle=0.5, metric='euclidean', verbose=0)

#### Plot variables by clusters

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_kmeans, 
    variable_name='torque_ds', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_kmeans, 
    variable_name='L1_2316', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

In [ ]:
plot_variable_by_cluster(
    df_data=df_dataset_anomalies, 
    labels=labels_kmeans, 
    variable_name='PIC2316_CV', # nome da variavel
    timestamp_col=timestamp # variável de timestamp
)

In [ ]:
cluster_colors = {
    0: 'rgba(31, 119, 180, 1)',   # Azul Forte
    1: 'rgba(255, 127, 14, 1)',   # Laranja Vivo
    2: 'rgba(44, 160, 44, 1)',    # Verde Floresta
    3: 'rgba(214, 39, 40, 1)',    # Vermelho Tijolo
    4: 'rgba(148, 103, 189, 1)',  # Roxo
    5: 'rgba(140, 86, 75, 1)',    # Marrom
    6: 'rgba(227, 119, 194, 1)',  # Rosa
    7: 'rgba(127, 127, 127, 1)',  # Cinza
    8: 'rgba(188, 189, 34, 1)',   # Verde Oliva
    9: 'rgba(23, 190, 207, 1)'    # Azul Teal/Ciano
}

for (start, end), label in zip(anomaly_periods, labels_hierarchical):
    color = cluster_colors.get(label, 'rgba(128, 128, 128, 0.3)')
    
    fig_all_period.add_vrect(
        x0=start, 
        x1=end,
        fillcolor=color,
        opacity=1,           # Opacidade já controlada na cor (rgba) ou use float aqui se usar hex
        layer="below",       # Coloca a faixa atrás das linhas do gráfico
        line_width=0,        # Remove borda da faixa
    )

fig_all_period.show()